# Exploration of Supreme Court pages from wikipedia

In [27]:
import sys
import httpx
import lxml.html
import time
import csv
import pandas as pd

In [28]:
# 122 pa3
ALLOWED_DOMAIN = "https://en.wikipedia.org/wiki/"


def make_link_absolute(path):
    """
    Given a relative URL like "/abc/def" or "?page=2"
    and a complete URL like "https://example.com/1/2/3" this function will
    combine the two yielding a URL like "https://example.com/abc/def"

    Parameters:
        * rel_url:      a URL or fragment
        * current_url:  a complete URL used to make the request that
                        contained a link to rel_url

    Returns:
        A full URL with protocol & domain that refers to rel_url.
    """

    return ALLOWED_DOMAIN + path

In [29]:
# 122 pa3
REQUEST_DELAY = 0.1


def make_request(url):
    """
    Make a request to `url` and return the raw response.

    This function ensure that the domain matches what is expected
    and that the rate limit is obeyed.
    """

    # check if URL starts with an allowed domain name
    for domain in ALLOWED_DOMAIN:
        if url.startswith(domain):
            break
    else:
        # note: this else is indented correctly, it is a less-commonly used
        # for-else statement.  the condition is only met if the for loop
        # *never* breaks, i.e. no domains match
        raise ValueError(f"can not fetch {url}, must be in {ALLOWED_DOMAIN}")
    time.sleep(REQUEST_DELAY)
    print(f"Fetching {url}")

    # https://www.scraperapi.com/blog/headers-and-cookies-for-web-scraping/
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/101.0.4951.64 Safari/537.36 Edg/101.0.1210.47"
        )
    }

    resp = httpx.get(url, headers=headers)
    resp.raise_for_status()
    return resp

In [30]:
# 122 web scrpaing lab
def get_parse_html(url):
    """
    Return the HTML for a given committee from the FEC website.
    """

    resp = make_request(url)
    html = resp.text
    root = lxml.html.fromstring(html)

    return root

In [31]:
def scrape_page(root, state):
    """
    Given the html text of a page, extract the info from the table of judges.
    """

    # Get the entire table, headers (th) and data (td)
    table = root.cssselect("table.wikitable.sortable tbody")[0]

    # Extract just the header values
    header_row = table.cssselect("tr th")
    header = []
    for th in header_row:
        text = th.text_content().strip()
        header.append(text)

    # Make a data column for state
    header.append("State")

    # Extract the table data
    rows = []
    for tr in table.cssselect("tr"):
        cells = tr.cssselect("td")
        if not cells:
            continue  # skip the header row

        row = []
        for td in cells:
            row.append(td.text_content().strip())

        # Add given state for each row
        row.append(state)

        rows.append(row)

    return header, rows

In [32]:
def run_scraper(state, path):
    url = make_link_absolute(path)
    root = get_parse_html(url)
    return scrape_page(root, state)

## Trying it out on Alabama

In [33]:
run_scraper("Alabama", "Alabama_Supreme_Court")

Fetching https://en.wikipedia.org/wiki/Alabama_Supreme_Court


(['Seat',
  'Name',
  'Born',
  'Start',
  'Term ends',
  'Party',
  'Appointer',
  'Law School',
  'State'],
 [['Chief Justice',
   'Sarah Hicks Stewart',
   '(1963-04-26) April 26, 1963 (age\xa063)',
   'January 11, 2019[a]',
   '2030',
   'Republican',
   '—.mw-parser-output .sr-only{border:0;clip:rect(0,0,0,0);clip-path:polygon(0px 0px,0px 0px,0px 0px);height:1px;margin:-1px;overflow:hidden;padding:0;position:absolute;width:1px;white-space:nowrap}N/a[b]',
   'Vanderbilt',
   'Alabama'],
  ['7',
   'Greg Shaw',
   '(1957-03-21) March 21, 1957 (age\xa069)',
   'January 20, 2009',
   '2026',
   'Republican',
   '—N/a[b]',
   'Samford',
   'Alabama'],
  ['6',
   'Alisa Kelli Wise',
   '(1962-12-14) December 14, 1962 (age\xa063)',
   'January 17, 2011',
   '2028',
   'Republican',
   '—N/a[b]',
   'Faulkner',
   'Alabama'],
  ['2',
   'Tommy Bryan',
   '(1956-05-16) May 16, 1956 (age\xa069)',
   'January 11, 2013',
   '2030',
   'Republican',
   '—N/a[b]',
   'Faulkner',
   'Alabama'],


## All states

In [34]:
def make_path(state):
    fix = state.replace(" ", "_")
    return f"{fix}_Supreme_Court"

In [39]:
states = ["Alabama", "Alaska", "Arizona", "Arkansas", "North Carolina", "Vermont"]

states_to_scrape = {state: make_path(state) for state in states}

In [40]:
all_dfs = []

for state, path in states_to_scrape.items():
    header, rows = run_scraper(state, path)
    df = pd.DataFrame(rows, columns=header)
    all_dfs.append(df)

big_df = pd.concat(all_dfs, ignore_index=True)

Fetching https://en.wikipedia.org/wiki/Alabama_Supreme_Court
Fetching https://en.wikipedia.org/wiki/Alaska_Supreme_Court
Fetching https://en.wikipedia.org/wiki/Arizona_Supreme_Court
Fetching https://en.wikipedia.org/wiki/Arkansas_Supreme_Court
Fetching https://en.wikipedia.org/wiki/North_Carolina_Supreme_Court
Fetching https://en.wikipedia.org/wiki/Vermont_Supreme_Court


In [37]:
big_df

,Seat,Name,Born,Start,Term ends,Party,Appointer,Law School,State,Chief term,Mandatory retirement[a],Law school,Term ends[a],Mandatory retirement,Name[24],Mandatory retirement[b]
0,Chief Justice,Sarah Hicks Stewart,"(1963-04-26) April 26, 1963 (age 63)","January 11, 2019[a]",2030,Republican,—.mw-parser-output .sr-only{border:0;clip:rect...,Vanderbilt,Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7,Greg Shaw,"(1957-03-21) March 21, 1957 (age 69)","January 20, 2009",2026,Republican,—N/a[b],Samford,Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6,Alisa Kelli Wise,"(1962-12-14) December 14, 1962 (age 63)","January 17, 2011",2028,Republican,—N/a[b],Faulkner,Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,Tommy Bryan,"(1956-05-16) May 16, 1956 (age 69)","January 11, 2013",2030,Republican,—N/a[b],Faulkner,Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,Will Sellers,"(1963-02-10) February 10, 1963 (age 63)","May 25, 2017",2030,Republican,Kay Ivey (R),Alabama,Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,8,Brady E. Mendheim Jr.,"(1968-07-26) July 26, 1968 (age 57)","January 15, 2019[c]",2026,Republican,Kay Ivey (R),Samford,Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,5,Greg Cook,1962 or 1963 (age 62–63),"January 17, 2023",2028,Republican,—N/a[b],Harvard,Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,Chris McCool,1967 or 1968 (age 57–58),"January 24, 2025",2030,Republican,—N/a[b],Alabama,Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,4,Will Parker,,"November 10, 2025",2026,Republican,Kay Ivey (R),Alabama,Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,"Susan M. Carney, Chief Justice","(1961-06-17) June 17, 1961 (age 64)","August 26, 2016",2030,NaN,Bill Walker (I),NaN,Alaska,2025–present,2031,Harvard,NaN,NaN,NaN,NaN


In [ ]:
# big_df.to_csv("all_states_justices.csv", index=False)
